<a href="https://colab.research.google.com/github/nicolasengland14-gif/ECON5200-Applied-Data-Analytics-in-Economics/blob/main/Lab%2013/Lab_13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

# Step 1: Ingestion and Naive Model
url = 'https://raw.githubusercontent.com/nicolasengland14-gif/ECON5200-Applied-Data-Analytics-in-Economics/refs/heads/main/Data/Zillow_California_2026_Hedonic.csv'
df = pd.read_csv(url)

naive_model = smf.ols('Sale_Price ~ Property_Age', data=df).fit()
print(naive_model.summary())
print("\nNaive Age Coefficient:", naive_model.params['Property_Age'])

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.757
Model:                            OLS   Adj. R-squared:                  0.757
Method:                 Least Squares   F-statistic:                     3105.
Date:                Fri, 13 Mar 2026   Prob (F-statistic):          1.26e-308
Time:                        18:28:05   Log-Likelihood:                -12818.
No. Observations:                1000   AIC:                         2.564e+04
Df Residuals:                     998   BIC:                         2.565e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     3.013e+05   7218.570     41.742   

In [2]:
df

,Property_Age,Distance_to_Tech_Hub,Sale_Price
0,77.5,38.1,684100.56
1,11.0,95.1,413634.22
2,47.7,73.5,456709.35
3,61.9,60.3,624533.95
4,100.8,16.4,870137.54
...,...,...,...
995,87.7,10.1,932592.35
996,21.2,91.8,412741.12
997,96.5,14.5,880901.56
998,20.1,95.1,396659.79


In [3]:
# Step 2: The Multivariate Model
multi_model = smf.ols('Sale_Price ~ Property_Age + 	Distance_to_Tech_Hub', data=df).fit()
print(multi_model.summary())
print("\nMultivariate Age Coefficient:", multi_model.params['Property_Age'])

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.954
Model:                            OLS   Adj. R-squared:                  0.954
Method:                 Least Squares   F-statistic:                 1.040e+04
Date:                Fri, 13 Mar 2026   Prob (F-statistic):               0.00
Time:                        18:33:03   Log-Likelihood:                -11982.
No. Observations:                1000   AIC:                         2.397e+04
Df Residuals:                     997   BIC:                         2.399e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept             1.203e+06 

In [4]:
# Step 3: FWL Theorem Manual Proof
# 3a: Partial out distance from Price
res_y_model = smf.ols('Sale_Price ~ Distance_to_Tech_Hub', data=df).fit()
df['Price_Residuals'] = res_y_model.resid

In [5]:
# 3b: Partial out distance from Age
res_x_model = smf.ols('Property_Age ~ Distance_to_Tech_Hub', data=df).fit()
df['Age_Residuals'] = res_x_model.resid

In [6]:
# 3c: Regress Residuals on Residuals (-1 removes the intercept for exact mathematical matching)
fwl_model = smf.ols('Price_Residuals ~ Age_Residuals - 1', data=df).fit()
print("\nFWL Isolated Age Coefficient:", fwl_model.params['Age_Residuals'])


FWL Isolated Age Coefficient: -2063.129216802139


In [7]:
"""
OLS Multivariate Regression — 3D Hyperplane Visualization
==========================================================
Predicts Sale_Price from Property_Age and Distance_to_Tech_Hub
using statsmodels OLS, then renders the fitted 2D regression
hyperplane alongside raw data points via plotly.graph_objects.
"""

import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.graph_objects as go

# ── 1. Simulate (or load) your dataset ──────────────────────────────────────
# Replace this block with:  df = pd.read_csv("your_data.csv")
np.random.seed(42)
n = 300

property_age       = np.random.uniform(1, 50, n)          # years
distance_tech_hub  = np.random.uniform(0.5, 30, n)        # km

# True data-generating process (with noise)
sale_price = (
    500_000
    - 3_200  * property_age
    - 8_500  * distance_tech_hub
    + np.random.normal(0, 25_000, n)
)

df = pd.DataFrame({
    "Sale_Price":          sale_price,
    "Property_Age":        property_age,
    "Distance_to_Tech_Hub": distance_tech_hub,
})

# ── 2. Fit the OLS model with statsmodels ────────────────────────────────────
X = sm.add_constant(df[["Property_Age", "Distance_to_Tech_Hub"]])
#   ↑ add_constant prepends a column of 1s so statsmodels estimates β₀ (intercept)

y = df["Sale_Price"]

model  = sm.OLS(y, X).fit()
print(model.summary())

# ── 3. Extract OLS coefficients from the fitted results object ───────────────
# model.params is a pandas Series indexed by variable name:
#   model.params["const"]                  → β₀  (intercept)
#   model.params["Property_Age"]           → β₁
#   model.params["Distance_to_Tech_Hub"]   → β₂
beta_0 = model.params["const"]
beta_1 = model.params["Property_Age"]
beta_2 = model.params["Distance_to_Tech_Hub"]

print(f"\nExtracted coefficients:")
print(f"  β₀ (intercept)           = {beta_0:,.0f}")
print(f"  β₁ (Property_Age)        = {beta_1:,.0f}")
print(f"  β₂ (Distance_to_Tech_Hub)= {beta_2:,.0f}")

# ── 4. Build a meshgrid spanning the observed ranges of both predictors ───────
# np.linspace creates 50 evenly-spaced values across each predictor's range.
# np.meshgrid turns those two 1-D arrays into two 2-D arrays (age_grid, dist_grid)
# so that every (row, col) combination represents one (age, distance) pair on the plane.

age_vals  = np.linspace(df["Property_Age"].min(),        df["Property_Age"].max(),        50)
dist_vals = np.linspace(df["Distance_to_Tech_Hub"].min(), df["Distance_to_Tech_Hub"].max(), 50)

age_grid, dist_grid = np.meshgrid(age_vals, dist_vals)
#   age_grid  shape: (50, 50) — age value at each grid point
#   dist_grid shape: (50, 50) — distance value at each grid point

# ── 5. Evaluate the OLS plane equation at every grid point ───────────────────
# ŷ = β₀ + β₁·age + β₂·distance
# Broadcasting applies the scalar coefficients element-wise across the 2-D grids.
price_grid = beta_0 + beta_1 * age_grid + beta_2 * dist_grid
#   price_grid shape: (50, 50) — predicted Sale_Price at each grid point

# ── 6. Build the Plotly 3D figure ────────────────────────────────────────────
fig = go.Figure()

# — Scatter: raw observed data points —
fig.add_trace(go.Scatter3d(
    x=df["Property_Age"],
    y=df["Distance_to_Tech_Hub"],
    z=df["Sale_Price"],
    mode="markers",
    marker=dict(
        size=3.5,
        color=df["Sale_Price"],        # colour-encode Sale_Price for depth cue
        colorscale="Plasma",
        opacity=0.75,
        colorbar=dict(title="Sale Price ($)", x=1.02),
    ),
    name="Observed data",
    hovertemplate=(
        "<b>Property Age:</b> %{x:.1f} yrs<br>"
        "<b>Distance:</b> %{y:.1f} km<br>"
        "<b>Sale Price:</b> $%{z:,.0f}<extra></extra>"
    ),
))

# — Surface: fitted OLS regression hyperplane —
fig.add_trace(go.Surface(
    x=age_grid,          # 2-D array of Property_Age values on the grid
    y=dist_grid,         # 2-D array of Distance values on the grid
    z=price_grid,        # 2-D array of predicted Sale_Price values (the plane)
    colorscale="Blues",
    opacity=0.45,
    showscale=False,
    name="OLS hyperplane",
    hovertemplate=(
        "Age: %{x:.1f} yrs | Dist: %{y:.1f} km<br>"
        "Predicted: $%{z:,.0f}<extra>OLS Plane</extra>"
    ),
))

# ── 7. Layout styling ─────────────────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text="Multivariate OLS — Sale Price vs Property Age & Distance to Tech Hub",
        font=dict(size=16),
    ),
    scene=dict(
        xaxis_title="Property Age (years)",
        yaxis_title="Distance to Tech Hub (km)",
        zaxis_title="Sale Price ($)",
        xaxis=dict(backgroundcolor="#f7f7f7"),
        yaxis=dict(backgroundcolor="#f0f0f0"),
        zaxis=dict(backgroundcolor="#e8e8e8"),
    ),
    margin=dict(l=0, r=0, t=60, b=0),
    legend=dict(x=0.01, y=0.99),
    width=1000,
    height=700,
)

fig.write_html("ols_3d_regression.html", include_plotlyjs="cdn")
print("\n✅  Saved → ols_3d_regression.html")
fig.show()

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.923
Model:                            OLS   Adj. R-squared:                  0.922
Method:                 Least Squares   F-statistic:                     1771.
Date:                Fri, 13 Mar 2026   Prob (F-statistic):          9.00e-166
Time:                        18:39:54   Log-Likelihood:                -3463.8
No. Observations:                 300   AIC:                             6934.
Df Residuals:                     297   BIC:                             6945.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                  5.03e+05 